# 05 — Baseline Models

## Goal
Interpretable baselines: Dummy (benchmark) and Logistic Regression (probability output, finance-friendly coefficients).

In [ ]:
from pathlib import Path
import sys

# Locate the project root (works whether the notebook is run from notebooks/ or root)
root = Path.cwd()
while not (root / "src" / "data_loader.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("project root:", root)


In [ ]:
from pathlib import Path

missing = [p.name for p in [
    root / "models" / "metadata.json",
    root / "reports" / "model_results.csv",
] if not p.exists()]
if missing:
    print("Missing artifacts:", missing)
    print("Run first:  python -m src.train && python -m src.evaluate && python -m src.explain")
else:
    print("All artifacts present.")


In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from src.train import prepare_matrices, lr_pipeline
from sklearn.dummy import DummyClassifier

data = prepare_matrices()
dummy = DummyClassifier(strategy="prior").fit(data["X_train"], data["y_train"])
logit = lr_pipeline("none", data["numeric_cols"]).fit(data["X_train"], data["y_train"])

rows = []
for name, model in [("dummy", dummy), ("logistic", logit)]:
    p = model.predict_proba(data["X_val"])[:, 1]
    rows.append({"model": name,
                 "val_roc_auc": roc_auc_score(data["y_val"], p),
                 "val_pr_auc": average_precision_score(data["y_val"], p)})
pd.DataFrame(rows).round(4)


Logistic regression already reaches ROC-AUC ≈ 0.75 / PR-AUC ≈ 0.52 on validation — a strong, fully interpretable baseline. The research question becomes: *how much more do tree ensembles buy?*